# openevals smoke test

Verifies that `openevals` (LLM evaluation toolkit) imports cleanly and exposes its primary API surface.

In [ ]:
# Example 1: Public API surface — verify core evaluator-factory entry points exist.
import openevals

expected_names = [
    'create_llm_as_judge',
    'create_async_llm_as_judge',
]

present = {n: hasattr(openevals, n) for n in expected_names}
for name, exists in present.items():
    obj = getattr(openevals, name, None)
    print(f"{name}: present={exists}, type={type(obj).__name__}")

assert any(present.values()), f"No expected openevals public APIs found; got {dir(openevals)[:30]}"

In [ ]:
# Example 2: Prompts subpackage — canned evaluator prompts are importable strings.
from openevals import prompts

prompt_names = [n for n in dir(prompts) if not n.startswith('_') and n.isupper()]
print(f"Found {len(prompt_names)} prompt constant(s) in openevals.prompts")
for name in prompt_names[:5]:
    val = getattr(prompts, name)
    snippet = (val[:80] + '...') if isinstance(val, str) and len(val) > 80 else val
    print(f"  {name}: {type(val).__name__} = {snippet!r}")

assert len(prompt_names) > 0 or hasattr(prompts, '__all__'), "openevals.prompts has no constants"

In [ ]:
# Example 3: Types & schema — EvaluatorResult / SimpleEvaluator are part of the public typing surface.
from openevals import types as oe_types

type_names = [n for n in dir(oe_types) if not n.startswith('_')]
print(f"openevals.types exports {len(type_names)} name(s)")
for n in type_names[:10]:
    obj = getattr(oe_types, n)
    print(f"  {n}: {type(obj).__name__}")

# EvaluatorResult is typically a TypedDict — must be subscriptable / introspectable.
if hasattr(oe_types, 'EvaluatorResult'):
    er = oe_types.EvaluatorResult
    print(f"EvaluatorResult: {er}")
assert len(type_names) > 0, "openevals.types is empty"

In [ ]:
# Example 4: Exact-match style evaluator — deterministic, no network needed.
import openevals

# Look for a string/exact-match evaluator factory under common names.
candidates = [
    'create_exact_match_evaluator',
    'exact_match',
    'create_json_match_evaluator',
]
found = [n for n in candidates if hasattr(openevals, n)]
print(f"Available deterministic-evaluator factories: {found}")

# The string subpackage holds the string-distance evaluators.
from openevals import string as oe_string
print(f"openevals.string exports: {[n for n in dir(oe_string) if not n.startswith('_')][:15]}")

# Run the deterministic evaluators (no LLM, no network).
match = openevals.exact_match(outputs={"answer": 42}, reference_outputs={"answer": 42})
mismatch = openevals.exact_match(outputs={"answer": 42}, reference_outputs={"answer": 41})
print(f"exact_match: {match['score']} / {mismatch['score']}")
assert (match["key"], match["score"], mismatch["score"]) == ("exact_match", True, False), (match, mismatch)

distance = oe_string.levenshtein_distance(outputs="kitten", reference_outputs="sitting")
print(f"levenshtein_distance: {distance['score']:.4f}")
# levenshtein_distance returns a similarity in [0, 1] (1 - edits / max length), not the raw edit count
assert abs(distance["score"] - (1 - 3 / 7)) < 1e-9, distance

In [ ]:
# Example 5: create_llm_as_judge factory is callable and validates its signature.
import inspect

from openevals import create_llm_as_judge

sig = inspect.signature(create_llm_as_judge)
params = list(sig.parameters.keys())
print(f"create_llm_as_judge signature: {sig}")
print(f"params: {params}")

# The factory should accept at least a 'prompt' (or 'judge'/'model') argument.
assert any(p in params for p in ('prompt', 'judge', 'model')), (
    f"create_llm_as_judge missing expected param; got {params}"
)
print("\nopenevals smoke test PASSED")